# 🔩 Dashboard de Materias Primas Estratégicas para la Industria Automotriz y EV

**Autor:** Enrique H.G. | Portafolio GitHub

Este dashboard consulta en tiempo real (vía API pública de Yahoo Finance) los precios de futuros y ETFs de los materiales críticos para la manufactura automotriz y de vehículos eléctricos:

| Material | Uso principal en la industria |
|---|---|
| Cobre (`HG=F`) | Cableado, motores eléctricos, bobinados |
| Aluminio (`ALI=F`) | Chasis y carrocerías ligeras |
| Paladio (`PA=F`) | Catalizadores de motores a gasolina |
| Platino (`PL=F`) | Catalizadores de motores diésel |
| Litio y Baterías (`LIT` ETF) | Celdas de batería para EV |

Este panel se enfoca en **commodities de manufactura**, relevantes para análisis de costos de producción, cadenas de suministro y decisiones de ingeniería de materiales.

> Ejecutar las celdas en orden. Todas las variables persisten en el kernel, por lo que el panel permanece interactivo después de correr la última celda.

In [1]:
# Importación de librerías y configuración del entorno
import warnings
warnings.filterwarnings("ignore")

import yfinance as yf
import pandas as pd
import numpy as np

import plotly.graph_objects as go
import plotly.io as pio

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Fuerza el renderizado de gráficos Plotly dentro de las celdas del notebook
pio.renderers.default = "notebook"

print("✅ Librerías cargadas correctamente. Entorno listo para Jupyter Notebook.")

✅ Librerías cargadas correctamente. Entorno listo para Jupyter Notebook.


In [2]:
# Capa de datos — consulta a la API pública de Yahoo Finance con caché y manejo de errores

MATERIALES_AUTOMOTRICES = {
    "Cobre — Cableado / Motores EV": "HG=F",
    "Aluminio — Chasis ligero": "ALI=F",
    "Paladio — Catalizadores gasolina": "PA=F",
    "Platino — Catalizadores diésel": "PL=F",
    "Litio y Baterías (ETF LIT)": "LIT",
}

# Caché en memoria: evita volver a golpear la API si ya se consultó esa combinación
_CACHE_MERCADO = {}

def obtener_datos_mercado(tickers, periodo, intervalo):
    '''Descarga precios históricos desde la API de Yahoo Finance (vía yfinance).

    Devuelve (dataframe, None) si tiene éxito, o (None, mensaje_error) si falla.
    '''
    clave = (tuple(sorted(tickers)), periodo, intervalo)
    if clave in _CACHE_MERCADO:
        return _CACHE_MERCADO[clave], None

    try:
        datos = yf.download(
            tickers=tickers,
            period=periodo,
            interval=intervalo,
            group_by="ticker",
            progress=False,
            threads=True,
        )
        if datos is None or datos.empty:
            raise ValueError(
                "La API respondió sin datos para esta combinación de periodo/intervalo. "
                "Prueba con un periodo más amplio."
            )
        _CACHE_MERCADO[clave] = datos
        return datos, None

    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

print(f"✅ Función de consulta lista. Materiales disponibles: {len(MATERIALES_AUTOMOTRICES)}")

✅ Función de consulta lista. Materiales disponibles: 5


In [3]:
# Controles interactivos (ipywidgets) y función de graficado (Plotly)

nombres_materiales = list(MATERIALES_AUTOMOTRICES.keys())

selector_materiales = widgets.SelectMultiple(
    options=nombres_materiales,
    value=(nombres_materiales[0], nombres_materiales[2]),
    description="Materiales:",
    rows=5,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

selector_periodo = widgets.Dropdown(
    options=[("1 mes", "1mo"), ("3 meses", "3mo"), ("6 meses", "6mo"), ("1 año", "1y"), ("2 años", "2y")],
    value="6mo",
    description="Periodo:",
)

selector_intervalo = widgets.Dropdown(
    options=[("Diario", "1d"), ("Semanal", "1wk")],
    value="1d",
    description="Intervalo:",
)

toggle_vista = widgets.ToggleButtons(
    options=[("Precio absoluto (USD)", False), ("Variación % · Base 100", True)],
    description="Vista:",
    style={"description_width": "initial"},
)

check_media_movil = widgets.Checkbox(value=False, description="Mostrar media móvil")

slider_ventana_mm = widgets.IntSlider(
    value=7, min=2, max=30, step=1,
    description="Ventana MM (periodos):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="380px"),
)


def graficar_dashboard(materiales_sel, periodo, intervalo, normalizar, mostrar_mm, ventana_mm):
    if len(materiales_sel) == 0:
        display(Markdown("⚠️ **Selecciona al menos un material** en la lista para generar el gráfico."))
        return

    tickers = [MATERIALES_AUTOMOTRICES[m] for m in materiales_sel]
    datos, error = obtener_datos_mercado(tickers, periodo, intervalo)

    if error:
        mensaje_error = (
            "### ⚠️ No fue posible consultar la API de Yahoo Finance\n\n"
            "```\n" + error + "\n```\n\n"
            "Sugerencias: verifica tu conexión a internet, prueba con otro periodo/intervalo, "
            "o vuelve a intentarlo en unos segundos (la API pública puede aplicar límites de uso)."
        )
        display(Markdown(mensaje_error))
        return

    fig = go.Figure()

    for nombre, ticker in zip(materiales_sel, tickers):
        try:
            serie = datos["Close"] if len(tickers) == 1 else datos[ticker]["Close"]
            serie = serie.dropna()
            if serie.empty:
                continue

            y = (serie / serie.iloc[0] * 100.0) if normalizar else serie
            fig.add_trace(go.Scatter(x=serie.index, y=y, mode="lines", name=nombre, line=dict(width=2)))

            if mostrar_mm:
                mm = y.rolling(window=ventana_mm).mean()
                fig.add_trace(go.Scatter(
                    x=serie.index, y=mm, mode="lines",
                    name=f"{nombre} · MM{ventana_mm}",
                    line=dict(width=1.5, dash="dot"),
                ))
        except Exception as e:
            print(f"⚠️ No se pudo procesar '{nombre}' ({ticker}): {e}")

    titulo_eje_y = "Variación (%) · Base 100" if normalizar else "Precio (USD)"

    fig.update_layout(
        title="Materias Primas Clave — Manufactura Automotriz y Vehículos Eléctricos",
        xaxis_title="Fecha",
        yaxis_title=titulo_eje_y,
        template="plotly_white",
        height=520,
        legend_title="Material",
        hovermode="x unified",
        margin=dict(t=70, b=40),
    )
    fig.update_xaxes(rangeslider_visible=True)
    fig.show()


panel_interactivo = widgets.interactive(
    graficar_dashboard,
    materiales_sel=selector_materiales,
    periodo=selector_periodo,
    intervalo=selector_intervalo,
    normalizar=toggle_vista,
    mostrar_mm=check_media_movil,
    ventana_mm=slider_ventana_mm,
)

print("✅ Controles construidos. Ejecuta la siguiente celda para mostrar el dashboard.")

✅ Controles construidos. Ejecuta la siguiente celda para mostrar el dashboard.


In [4]:
# Renderizado del dashboard completo

display(Markdown("## 📊 Panel interactivo — mueve los controles para actualizar el gráfico"))
display(panel_interactivo)

## 📊 Panel interactivo — mueve los controles para actualizar el gráfico

interactive(children=(SelectMultiple(description='Materiales:', index=(0, 2), layout=Layout(width='420px'), op…